# Import a fine-tuned model

Helper notebook, not part of the lab. It copies a published checkpoint into your own
bucket and registers it in the Model Package Group, so you can run
3-dpo-evaluation.ipynb and 4-dpo-deployment.ipynb without training first.

Skip this notebook if you trained your own model.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:   {role}")
print(f"bucket: {bucket_name}")
print(f"region: {region}")

In [ ]:
# The workshop assets are replicated per region, and Studio's VPC endpoint refuses
# cross-region S3 requests, so read from the replica in this session's region.
ASSET_BUCKETS = {
    "us-east-1": "ws-assets-prod-iad-r-iad-ed304a55c2ca1aee",
    "us-west-2": "ws-assets-prod-iad-r-pdx-f3b3f9f1a7d6a3d0",
}
asset_region = boto3.Session().region_name
PUBLISHED_CHECKPOINT = (f"s3://{ASSET_BUCKETS[asset_region]}/"
                        "548b5be9-2da8-4c93-82f7-b0b474108ab3/checkpoints/lab2/hf_merged/")
RECIPE_NAME = "llmft_llama3_2_1b_instruct_seq4k_gpu_dpo"

base_model_id = "meta-textgeneration-llama-3-2-1b-instruct"

# the deployment notebook joins checkpoints/hf_merged onto the registered URI,
# so the copy keeps that layout
project_prefix = f"{base_model_id}-imported"
checkpoint_uri = f"s3://{bucket_name}/{project_prefix}/checkpoints/hf_merged/"
register_uri = f"s3://{bucket_name}/{project_prefix}/"

print(checkpoint_uri)

In [ ]:
import hashlib

MAX_MPG_NAME_LENGTH = 63
suffix = "-dpo"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    model_package_group_name = f"{base_model_id[:keep].rstrip('-')}-{digest}{suffix}"
else:
    model_package_group_name = candidate

print(model_package_group_name)

In [ ]:
import subprocess

# The published bucket allows reading a known key but not listing, so the files are
# named here rather than copied with --recursive. Each copy is server-side.
CHECKPOINT_FILES = [
    "model.safetensors",
    "config.json",
    "generation_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "chat_template.jinja",
]

for name in CHECKPOINT_FILES:
    # --copy-props none: the default asks the source for its tags, which the
    # published bucket does not allow
    subprocess.run(["aws", "s3", "cp", PUBLISHED_CHECKPOINT + name,
                    checkpoint_uri + name, "--copy-props", "none",
                    "--only-show-errors"], check=True)
    print(f"  {name}")

!aws s3 ls {checkpoint_uri} --summarize | tail -3

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

try:
    ModelPackageGroup.get(model_package_group_name=model_package_group_name)
    print(f"group exists: {model_package_group_name}")
except ClientError:
    ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Store models from SageMaker serverless DPO customization",
    )
    print(f"created group: {model_package_group_name}")

# read the current hub version rather than pinning one, which goes stale
hub_version = sm_client.describe_hub_content(
    HubName="SageMakerPublicHub", HubContentType="Model",
    HubContentName=base_model_id)["HubContentVersion"]

response = sm_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription="Pre-trained model imported from S3",
    InferenceSpecification={
        "Containers": [
            {
                "ModelDataSource": {
                    "S3DataSource": {
                        "S3Uri": register_uri,
                        "S3DataType": "S3Prefix",
                        "CompressionType": "None",
                    }
                },
                "BaseModel": {
                    "HubContentName": base_model_id,
                    "HubContentVersion": hub_version,
                    "RecipeName": RECIPE_NAME,
                },
            }
        ],
    },
    ModelApprovalStatus="Approved",
)

print(response["ModelPackageArn"])
print("\nDone. Continue with 3-dpo-evaluation.ipynb and 4-dpo-deployment.ipynb.")